# Lista 2: Inteligência Artificial (UFAL)
## Questão 9: Sistema de recomendação baseado em ontologia OWL 2

**Dupla:** Bruno Gomes de Barros Filho (21111360); Antônio Rodrigo Lima de Andrade Tenório (23210707)
**Disciplina:** ECOM031: Inteligência Artificial
**Professor:** Evandro de Barros Costa
**Período:** 2026.2

### Enunciado (Q9)
> Desenvolva um Sistema de Recomendação Baseado em Ontologias. O objetivo desta atividade é aplicar os
> conceitos de Representação do Conhecimento com Ontologias por meio do desenvolvimento de um sistema
> inteligente baseado em conhecimento utilizando a linguagem OWL 2.0 (Web Ontology Language).
>
> Requisitos mínimos: (1) ontologia com pelo menos 10 classes, 15 propriedades, hierarquia de classes,
> restrições e axiomas e indivíduos suficientes; (2) base de conhecimento que permita consultas semânticas
> e inferências por reasoner; (3) sistema que receba informações do usuário, consulte a ontologia, gere
> recomendações justificadas e exiba os elementos usados na decisão; (4) uso de reasoner OWL 2 (HermiT,
> Pellet ou FaCT++) para classificação automática de indivíduos, descoberta de relacionamentos implícitos,
> verificação de consistência e apoio à recomendação.

**Domínio escolhido:** recomendação de atividades físicas para moradores de Maceió. O usuário informa idade,
peso, altura, nível de atividade, experiência, objetivos, condições de saúde, orçamento e preferência de
ambiente. A ontologia descreve atividades, benefícios, objetivos, condições de saúde, partes do corpo e locais.
O relatório completo está no `README.md` desta pasta.

## 1. Construção e carga da ontologia

O script `ontologia_atividades.py` gera os três arquivos `.owl` (principal, cópia inconsistente e cópia com classe insatisfazível). Rodar de novo produz o mesmo grafo RDF, o que é verificado em `test_q09_ontologia.py`.

In [1]:
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import rdflib
from owlready2 import OwlReadyInconsistentOntologyError, ThingClass

import ontologia_atividades as oa
from figuras_ontologia import (AZUL, LARANJA, SUPERFICIE, TEXTO, TEXTO_2, figura_contraindicacoes,
                               figura_grafo_propriedades, _salvar)
from raciocinio_owl import carregar_ontologia, explicar_pellet, raciocinar_hermit
from recomendador_atividades import PerfilUsuario, formatar, recomendar
from sintaxe_owl import axiomas, manchester

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 120)
oa.main()

gravado: ontologia_atividades_fisicas.owl
gravado: ontologia_atividades_fisicas_inconsistente.owl
gravado: ontologia_atividades_fisicas_insatisfazivel.owl


In [2]:
world, onto = carregar_ontologia()
classes = list(onto.classes())
obj_props, data_props = list(onto.object_properties()), list(onto.data_properties())
definidas = [k for k in classes if k.equivalent_to]
print(f"Classes: {len(classes)} ({len(definidas)} definidas por equivalência, {len(classes) - len(definidas)} primitivas)")
print(f"Propriedades: {len(obj_props) + len(data_props)} ({len(obj_props)} de objeto, {len(data_props)} de dados) + 1 de anotação")
print(f"Indivíduos: {len(list(onto.individuals()))} (atividades: {len(onto.AtividadeFisica.instances())}, usuários: {len(onto.Usuario.instances())})")
g = rdflib.Graph().parse(oa.ARQUIVO_OWL, format="xml")
print(f"rdflib relê o arquivo RDF/XML: {len(g)} triplas")

Classes: 39 (22 definidas por equivalência, 17 primitivas)
Propriedades: 28 (19 de objeto, 9 de dados) + 1 de anotação
Indivíduos: 66 (atividades: 18, usuários: 5)
rdflib relê o arquivo RDF/XML: 1147 triplas


### 1.1 Classes e propriedades

As classes primitivas formam a taxonomia declarada; as definidas têm condições necessárias e suficientes e é o reasoner quem decide quais indivíduos pertencem a elas.

In [3]:
pd.DataFrame([{"classe": k.name, "rótulo": str(k.label[0]),
               "superclasse declarada": ", ".join(p.name for p in k.is_a if isinstance(p, ThingClass)),
               "tipo": "definida" if k.equivalent_to else "primitiva"} for k in classes])

,classe,rótulo,superclasse declarada,tipo
0,AtividadeFisica,Atividade física,Thing,primitiva
1,NivelDeImpacto,Nível de impacto,Thing,definida
2,NivelTecnico,Nível técnico,Thing,definida
3,Local,Local,Thing,primitiva
4,ParteDoCorpo,Parte do corpo,Thing,primitiva
5,CondicaoDeSaude,Condição de saúde,Thing,primitiva
6,Usuario,Usuário,Thing,primitiva
7,Beneficio,Benefício,Thing,primitiva
8,Objetivo,Objetivo,Thing,primitiva
9,AtividadeAerobica,Atividade aeróbica,AtividadeFisica,primitiva


In [4]:
XSD = {int: "xsd:integer", float: "xsd:decimal", str: "xsd:string"}

def caracteristicas(p):
    nomes = [t.__name__.replace("Property", "") for t in p.is_a if t.__name__ in ("FunctionalProperty", "TransitiveProperty")]
    if getattr(p, "inverse_property", None):
        nomes.append(f"inversa de {p.inverse_property.name}")
    if getattr(p, "property_chain", None):
        nomes += [" ∘ ".join(q.name for q in ch.properties) for ch in p.property_chain]
    return "; ".join(nomes)

pd.DataFrame([{"propriedade": p.name, "tipo": "objeto" if p in obj_props else "dados",
               "domínio": ", ".join(d.name for d in p.domain),
               "imagem": ", ".join(XSD.get(r) or r.name for r in p.range),
               "características": caracteristicas(p)} for p in obj_props + data_props])

,propriedade,tipo,domínio,imagem,características
0,temImpacto,objeto,AtividadeFisica,NivelDeImpacto,Functional
1,temNivelTecnico,objeto,AtividadeFisica,NivelTecnico,Functional
2,realizadaEm,objeto,AtividadeFisica,Local,inversa de sediaAtividade
3,sediaAtividade,objeto,,,inversa de realizadaEm
4,parteDe,objeto,ParteDoCorpo,ParteDoCorpo,Transitive
5,sobrecarrega,objeto,AtividadeFisica,ParteDoCorpo,sobrecarrega ∘ parteDe
6,afeta,objeto,CondicaoDeSaude,ParteDoCorpo,inversa de afetadaPor
7,afetadaPor,objeto,,,inversa de afeta
8,contraindicadaPara,objeto,AtividadeFisica,CondicaoDeSaude,inversa de contraindica; sobrecarrega ∘ afetadaPor
9,contraindica,objeto,,,inversa de contraindicadaPara


In [5]:
figura_grafo_propriedades(onto, "q09_grafo_propriedades.png")

PosixPath('/home/antonio/Documentos/ia/lista2-ia-ufal/src/q09_q15_ontologia/figures/q09_grafo_propriedades.png')

![grafo de propriedades](figures/q09_grafo_propriedades.png)

### 1.2 Axiomas de definição e cadeias (Manchester Syntax, como aparecem no Protégé)

In [6]:
tab = pd.DataFrame(axiomas(onto))
tab[tab["tipo"].isin(["equivalência", "cadeia"])][["tipo", "Manchester"]]

,tipo,Manchester
4,equivalência,"Class: NivelDeImpacto EquivalentTo: {impacto_baixo, impacto_moderado, impacto_alto}"
5,equivalência,"Class: NivelTecnico EquivalentTo: {nivel_basico, nivel_intermediario, nivel_avancado}"
17,equivalência,Class: AtividadeDeBaixoImpacto EquivalentTo: AtividadeFisica and (temImpacto value impacto_baixo)
19,equivalência,Class: AtividadeDeAltoImpacto EquivalentTo: AtividadeFisica and (temImpacto value impacto_alto)
21,equivalência,Class: AtividadeLeve EquivalentTo: AtividadeFisica and (gastoMET some decimal[< 3.0])
23,equivalência,"Class: AtividadeModerada EquivalentTo: AtividadeFisica and (gastoMET some decimal[>= 3.0, < 6.0])"
25,equivalência,Class: AtividadeVigorosa EquivalentTo: AtividadeFisica and (gastoMET some decimal[>= 6.0])
28,equivalência,Class: AtividadeAoArLivre EquivalentTo: AtividadeFisica and (realizadaEm some LocalAoArLivre)
30,equivalência,Class: AtividadeEmAmbienteFechado EquivalentTo: AtividadeFisica and (realizadaEm some LocalFechado)
32,equivalência,Class: AtividadeAquatica EquivalentTo: AtividadeFisica and (realizadaEm some LocalAquatico)


## 2. Inferência com o HermiT

O HermiT que vem no owlready2 (versão 1.3.8; o Protégé 5.6 traz a 1.4.3, com o mesmo algoritmo de hipertableau) é chamado com `infer_property_values=True`, que equivale a marcar no Protégé as inferências de *object property assertions*. Antes de rodar guardo os fatos declarados para comparar.

In [7]:
sobrecarga_declarada = {a: list(a.sobrecarrega) for a in onto.AtividadeFisica.instances()}
tipos_declarados = {i: {t for t in i.is_a if isinstance(t, ThingClass)} for i in onto.individuals()}
def n_pares(p):
    valores = (getattr(i, p.python_name) for i in onto.individuals())
    return sum(len(v) if isinstance(v, list) else int(v is not None) for v in valores)

declarados = {p.name: n_pares(p) for p in obj_props}
segundos = raciocinar_hermit(world)
print(f"HermiT: ontologia consistente, {segundos:.2f} s (inclui subir a JVM)")
print("Classes insatisfazíveis:", [k.name for k in world.inconsistent_classes() if k.name != "Nothing"] or "nenhuma")

HermiT: ontologia consistente, 0.86 s (inclui subir a JVM)
Classes insatisfazíveis: nenhuma


### 2.1 Classificação automática de indivíduos

Nenhuma atividade foi cadastrada como `AtividadeVigorosa`, `AtividadeAdequadaParaIdoso` etc. A matriz abaixo mostra a classificação feita pelo HermiT a partir de `gastoMET`, `temImpacto`, `temNivelTecnico`, locais e custos.

In [8]:
atividades = sorted(onto.AtividadeFisica.instances(), key=lambda a: a.name)
colunas = [k for k in definidas if issubclass(k, onto.AtividadeFisica)]
matriz = pd.DataFrame([[int(a in k.instances()) for k in colunas] for a in atividades],
                      index=[str(a.label[0]) for a in atividades], columns=[k.name.replace("Atividade", "") for k in colunas])
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(matriz.values, cmap=plt.matplotlib.colors.ListedColormap(["#eef0f2", AZUL]), aspect="auto")
ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=55, ha="right")
ax.set_yticks(range(len(matriz.index)), matriz.index)
ax.set_xticks([x - 0.5 for x in range(1, len(matriz.columns))], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, len(matriz.index))], minor=True)
ax.grid(which="minor", color=SUPERFICIE, lw=2)
ax.tick_params(which="both", length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("Classes definidas atribuídas pelo HermiT a cada atividade (azul = pertence)", loc="left")
_salvar(fig, "q09_classificacao_atividades.png")
print("Pertinências inferidas:", int(matriz.values.sum()))
matriz.sum().rename("atividades").to_frame().T

Pertinências inferidas: 103


,DeBaixoImpacto,DeAltoImpacto,Leve,Moderada,Vigorosa,AoArLivre,EmAmbienteFechado,Aquatica,EmGrupo,Gratuita,AdequadaParaIniciante,AdequadaParaSedentario,AdequadaParaIdoso,AdequadaParaObesidade
atividades,11,4,2,8,8,10,9,3,3,6,11,10,9,9


![classificação](figures/q09_classificacao_atividades.png)

Os usuários de exemplo também são classificados por restrições de tipo de dado (idade, IMC, minutos por semana, meses de prática):

In [9]:
linhas = []
for u in sorted(onto.Usuario.instances(), key=lambda u: u.name):
    inferidos = sorted(k.name for k in u.INDIRECT_is_a if isinstance(k, ThingClass) and k.equivalent_to)
    linhas.append({"usuário": u.name, "idade": u.idade, "IMC": u.imc, "min/sem": u.minutosAtivosPorSemana,
                   "meses": u.mesesDeExperiencia, "classes inferidas": ", ".join(inferidos) or "(só Usuario)"})
pd.DataFrame(linhas)

,usuário,idade,IMC,min/sem,meses,classes inferidas
0,carla,25,22.0,300,36,(só Usuario)
1,joao,34,31.5,30,2,"UsuarioComObesidade, UsuarioIniciante, UsuarioSedentario"
2,lucas,28,23.5,240,24,UsuarioSemCondicaoDeSaude
3,maria,67,27.0,60,0,"UsuarioComCondicaoArticular, UsuarioIdoso, UsuarioIniciante, UsuarioSedentario"
4,pedro,45,25.0,200,12,(só Usuario)


### 2.2 Descoberta de relacionamentos implícitos

As propriedades `sobrecarrega`, `contraindicadaPara`, `naoDevePraticar`, `atendeObjetivo` e `temAtividadeCandidata` têm cadeias de propriedades. A tabela compara o número de pares declarados com o número de pares que o HermiT devolve depois do raciocínio.

In [10]:
inferidos = {p.name: n_pares(p) for p in obj_props}
pd.DataFrame([{"propriedade": n, "declarados": declarados[n], "após HermiT": inferidos[n],
               "novos": inferidos[n] - declarados[n]} for n in declarados if inferidos[n] != declarados[n]])

,propriedade,declarados,após HermiT,novos
0,parteDe,5,6,1
1,sobrecarrega,15,37,22
2,contraindicadaPara,0,23,23
3,contraindica,0,23,23
4,naoDevePraticar,0,15,15
5,atendeObjetivo,0,51,51
6,atendidoPor,0,51,51
7,temAtividadeCandidata,0,52,52


In [11]:
figura_contraindicacoes(onto, [onto.maria, onto.pedro], sobrecarga_declarada, "q09_contraindicacoes_inferidas.png")

PosixPath('/home/antonio/Documentos/ia/lista2-ia-ufal/src/q09_q15_ontologia/figures/q09_contraindicacoes_inferidas.png')

![contraindicações](figures/q09_contraindicacoes_inferidas.png)

Pedro tem hérnia de disco, que afeta a coluna vertebral. Nenhuma atividade foi cadastrada como sobrecarregando a coluna vertebral: CrossFit e vôlei de praia sobrecarregam a coluna lombar, que é `parteDe` da coluna vertebral. A transitividade de `parteDe` e a cadeia `sobrecarrega ∘ parteDe ⊑ sobrecarrega` levam o reasoner à contraindicação.

## 3. Consultas semânticas (SPARQL)

As três primeiras consultas usam o motor SPARQL do owlready2 sobre o grafo já enriquecido pelo HermiT. A última usa o rdflib direto no arquivo `.owl`, sem reasoner, e precisa reproduzir parte da cadeia com um caminho de propriedades (`:sobrecarrega/:parteDe*`).

In [12]:
PFX = "PREFIX : <http://ic.ufal.br/ecom031/lista2/atividades-fisicas#> "
def tabela(linhas, colunas):
    return pd.DataFrame([[getattr(v, "name", v) for v in l] for l in linhas], columns=colunas)

tabela(world.sparql(PFX + """
SELECT ?atividade ?met ?custo WHERE {
  ?atividade a :AtividadeAdequadaParaIdoso ; :gastoMET ?met ; :custoMensal ?custo .
} ORDER BY ?met"""), ["atividade (inferida AdequadaParaIdoso)", "MET", "custo"])

,atividade (inferida AdequadaParaIdoso),MET,custo
0,alongamento,2.3,0.0
1,yoga,2.5,160.0
2,pilates,3.0,280.0
3,tai_chi_chuan,3.0,0.0
4,caminhada,3.5,0.0
5,musculacao,3.5,100.0
6,stand_up_paddle,5.0,150.0
7,hidroginastica,5.5,150.0
8,natacao,5.8,180.0


In [13]:
tabela(world.sparql(PFX + """
SELECT ?usuario ?atividade WHERE { ?usuario :naoDevePraticar ?atividade . } ORDER BY ?usuario ?atividade
"""), ["usuário", "não deve praticar (inferido)"])

,usuário,não deve praticar (inferido)
0,maria,corrida
1,maria,zumba
2,maria,crossfit
3,maria,futebol_society
4,maria,volei_de_praia
5,joao,corrida
6,joao,ciclismo_na_orla
7,joao,spinning
8,joao,zumba
9,joao,crossfit


In [14]:
tabela(world.sparql(PFX + """
SELECT ?beneficio (COUNT(?a) AS ?n) WHERE { ?a :promove ?beneficio . }
GROUP BY ?beneficio ORDER BY DESC(?n)"""), ["benefício", "atividades que promovem"])

,benefício,atividades que promovem
0,condicionamento_cardiorrespiratorio,11
1,gasto_calorico,11
2,forca_muscular,6
3,interacao_social,6
4,equilibrio,5
5,relaxamento,5
6,flexibilidade,4


In [15]:
q = PFX + """
SELECT ?atividade ?condicao WHERE {
  :pedro :temCondicao ?condicao . ?condicao :afeta ?parte .
  ?atividade :sobrecarrega/:parteDe* ?parte .
}"""
pd.DataFrame([(str(r.atividade).split("#")[1], str(r.condicao).split("#")[1]) for r in g.query(q)],
             columns=["atividade", "condição de Pedro"])

,atividade,condição de Pedro
0,crossfit,hernia_de_disco
1,volei_de_praia,hernia_de_disco


## 4. Verificação de consistência

A cópia `ontologia_atividades_fisicas_inconsistente.owl` tem um erro de cadastro: a corrida recebe também `temImpacto value impacto_baixo`. Como `temImpacto` é funcional e os níveis são declarados diferentes entre si (`AllDifferent`), não existe modelo para a ontologia.

In [16]:
w_ruim, _ = carregar_ontologia(oa.ARQUIVO_INCONSISTENTE)
try:
    raciocinar_hermit(w_ruim)
except OwlReadyInconsistentOntologyError as erro:
    print("HermiT:", type(erro).__name__)
print("Justificativa do Pellet:")
print(explicar_pellet("--inconsistent", caminho=oa.ARQUIVO_INCONSISTENTE))

HermiT: OwlReadyInconsistentOntologyError
Justificativa do Pellet:


Axiom: Thing subClassOf Nothing
1)   Functional temImpacto
     corrida temImpacto impacto_alto
     DifferentIndividuals(impacto_alto
                          impacto_baixo
                          impacto_moderado)
     corrida type temImpacto value impacto_baixo


## 5. Sistema de recomendação

`recomendar(perfil)` carrega o `.owl` num `World` novo, cria o indivíduo `usuario_consulta`, roda o HermiT e usa apenas o que foi inferido:

1. perfis do usuário (classes definidas como `UsuarioIdoso`) e, pela anotação `exigeAtividadeDoTipo`, a classe de atividades seguras que cada perfil exige (filtro obrigatório);
2. `naoDevePraticar` deduzida pelas cadeias (filtro obrigatório, com o caminho da dedução);
3. `temAtividadeCandidata` deduzida pela cadeia `temObjetivo ∘ atendidoPor` (conjunto de partida);
4. pontuação: +3 por objetivo atendido, +1 se a atividade é do ambiente preferido, +1 se cabe no orçamento e -2 se não cabe.

### 5.1 Dona Lúcia: 68 anos, artrose no joelho, quer prevenir quedas e socializar

In [17]:
lucia = PerfilUsuario(nome="Dona Lúcia", idade=68, peso_kg=72, altura_m=1.60, minutos_ativos_por_semana=40,
                      meses_de_experiencia=0, objetivos=["prevenir_quedas", "socializar"],
                      condicoes=["artrose_no_joelho"], orcamento_mensal=200, prefere_ambiente="ar livre")
rec_lucia = recomendar(lucia)
print(formatar(rec_lucia))

Usuário: Dona Lúcia
Fatos informados:
  idade = 68
  IMC = 28.1
  minutosAtivosPorSemana = 40
  mesesDeExperiencia = 0
  temObjetivo = prevenir_quedas, socializar
  temCondicao = artrose_no_joelho
  orçamento = R$ 200.00
  ambiente preferido = ar livre
Perfis inferidos pelo HermiT (0.87 s):
  UsuarioComCondicaoArticular ≡ Usuario and (temCondicao some CondicaoArticular)  |  temCondicao = Artrose no joelho
  UsuarioIdoso ≡ Usuario and (idade some integer[>= 60])  |  idade = 68
  UsuarioIniciante ≡ Usuario and (mesesDeExperiencia some integer[< 6])  |  mesesDeExperiencia = 0
  UsuarioSedentario ≡ Usuario and (minutosAtivosPorSemana some integer[< 150])  |  minutosAtivosPorSemana = 40
Exigências de segurança (anotação exigeAtividadeDoTipo):
  UsuarioIdoso → AtividadeAdequadaParaIdoso: AtividadeFisica and (temImpacto value impacto_baixo) and (gastoMET some decimal[< 6.0])
  UsuarioIniciante → AtividadeAdequadaParaIniciante: AtividadeFisica and (temNivelTecnico value nivel_basico)
  Usuario

### 5.2 Carlos: 38 anos, IMC 32, hipertensão não controlada e hérnia de disco

In [18]:
carlos = PerfilUsuario(nome="Carlos", idade=38, peso_kg=98, altura_m=1.75, minutos_ativos_por_semana=20,
                       meses_de_experiencia=1, objetivos=["perder_peso", "melhorar_condicionamento"],
                       condicoes=["hipertensao_nao_controlada", "hernia_de_disco"], orcamento_mensal=150,
                       prefere_ambiente="fechado")
rec_carlos = recomendar(carlos)
print(formatar(rec_carlos, limite=3))

Usuário: Carlos
Fatos informados:
  idade = 38
  IMC = 32.0
  minutosAtivosPorSemana = 20
  mesesDeExperiencia = 1
  temObjetivo = perder_peso, melhorar_condicionamento
  temCondicao = hipertensao_nao_controlada, hernia_de_disco
  orçamento = R$ 150.00
  ambiente preferido = fechado
Perfis inferidos pelo HermiT (0.93 s):
  UsuarioComObesidade ≡ Usuario and (imc some decimal[>= 30.0])  |  imc = 32.0
  UsuarioIniciante ≡ Usuario and (mesesDeExperiencia some integer[< 6])  |  mesesDeExperiencia = 1
  UsuarioSedentario ≡ Usuario and (minutosAtivosPorSemana some integer[< 150])  |  minutosAtivosPorSemana = 20
Exigências de segurança (anotação exigeAtividadeDoTipo):
  UsuarioComObesidade → AtividadeAdequadaParaObesidade: AtividadeFisica and (temImpacto value impacto_baixo) and (gastoMET some decimal[>= 3.0])
  UsuarioIniciante → AtividadeAdequadaParaIniciante: AtividadeFisica and (temNivelTecnico value nivel_basico)
  UsuarioSedentario → AtividadeAdequadaParaSedentario: AtividadeFisica and (

### 5.3 Ana: 26 anos, treina há 3 anos, declara não ter condições de saúde

Como Ana respondeu "nenhuma", o sistema acrescenta o axioma de fechamento `temCondicao max 0 CondicaoDeSaude` e o HermiT a classifica como `UsuarioSemCondicaoDeSaude`. Se ela não respondesse, o sistema avisaria que, pela hipótese de mundo aberto, nada pode ser concluído sobre condições não informadas.

In [19]:
ana = PerfilUsuario(nome="Ana", idade=26, peso_kg=60, altura_m=1.68, minutos_ativos_por_semana=300,
                    meses_de_experiencia=40, objetivos=["ganhar_forca", "socializar"], condicoes=[],
                    orcamento_mensal=400, prefere_ambiente="ar livre")
rec_ana = recomendar(ana)
print(formatar(rec_ana, limite=5))

Usuário: Ana
Fatos informados:
  idade = 26
  IMC = 21.3
  minutosAtivosPorSemana = 300
  mesesDeExperiencia = 40
  temObjetivo = ganhar_forca, socializar
  temCondicao = nenhuma
  orçamento = R$ 400.00
  ambiente preferido = ar livre
Perfis inferidos pelo HermiT (0.87 s):
  UsuarioSemCondicaoDeSaude ≡ Usuario and (temCondicao max 0 CondicaoDeSaude)  |  temCondicao = (nenhum)
Exigências de segurança (anotação exigeAtividadeDoTipo):
  (nenhuma)
Aviso: Usuário declarou não ter condições de saúde (axioma de fechamento temCondicao max 0).
Contraindicações deduzidas (naoDevePraticar):
  (nenhuma)
Recomendações (5 de 11 aptas):
  1. CrossFit (pontuação 7)
       +3 atende 'Ganhar força' (promove Força muscular; cadeia promove ∘ contribuiPara ⊑ atendeObjetivo)
       +3 atende 'Socializar' (promove Interação social; cadeia promove ∘ contribuiPara ⊑ atendeObjetivo)
       +1 custo R$ 300.00 cabe no orçamento
  2. Vôlei de praia (pontuação 5)
       +3 atende 'Socializar' (promove Interação soc

In [20]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True, layout="constrained")
for ax, rec in zip(eixos, [rec_lucia, rec_carlos, rec_ana]):
    itens = rec.recomendadas[:6][::-1]
    ax.barh([i.rotulo for i in itens], [i.pontuacao for i in itens], color=AZUL, height=0.6)
    for y, i in enumerate(itens):
        ax.text(i.pontuacao + 0.1, y, str(i.pontuacao), va="center", color=TEXTO, fontsize=9)
    ax.set_title(f"{rec.usuario}: {len(rec.recomendadas)} aptas, {len(rec.descartadas)} descartadas",
                 loc="left", fontsize=10)
    ax.set_ylim(len(itens) - 6.5, len(itens) - 0.5)  # mesma altura de barra nos três painéis
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=TEXTO_2)
    ax.grid(axis="x", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
eixos[1].set_xlabel("pontuação (somente atividades que passaram nos filtros de segurança)", color=TEXTO_2)
_salvar(fig, "q09_ranking_recomendacoes.png")

PosixPath('/home/antonio/Documentos/ia/lista2-ia-ufal/src/q09_q15_ontologia/figures/q09_ranking_recomendacoes.png')

![ranking](figures/q09_ranking_recomendacoes.png)

### 5.4 A justificativa reconstruída confere com a do reasoner

O recomendador monta a explicação da contraindicação a partir dos axiomas. Para conferir, o Pellet calcula a justificativa mínima do mesmo fato na ontologia salva (Maria tem o mesmo problema de Dona Lúcia). É o recurso que o Protégé mostra no botão "?" ao lado de cada inferência.

In [21]:
print(explicar_pellet("--property-value", "maria,naoDevePraticar,zumba"))
print()
print(explicar_pellet("--property-value", "joao,naoDevePraticar,spinning"))
print()
print(explicar_pellet("--instance", "tai_chi_chuan,AtividadeAdequadaParaIdoso"))

Axiom: maria naoDevePraticar zumba
1)   afeta inverseOf afetadaPor
     maria temCondicao artrose_no_joelho
     zumba sobrecarrega joelho
     artrose_no_joelho afeta joelho
     temCondicao o contraindica subPropertyOf naoDevePraticar
     contraindicadaPara inverseOf contraindica
     sobrecarrega o afetadaPor subPropertyOf contraindicadaPara



Axiom: joao naoDevePraticar spinning
1)   afeta inverseOf afetadaPor
     AtividadeVigorosa equivalentTo AtividadeFisica
                                    and gastoMET some decimal[>= 6.0]
     joao temCondicao hipertensao_nao_controlada
     hipertensao_nao_controlada afeta sistema_cardiovascular
     temImpacto domain AtividadeFisica
     spinning gastoMET 8.5
     AtividadeVigorosa subClassOf sobrecarrega value sistema_cardiovascular
     temCondicao o contraindica subPropertyOf naoDevePraticar
     spinning temImpacto impacto_baixo
     sobrecarrega o afetadaPor subPropertyOf contraindicadaPara
     contraindicadaPara inverseOf contraindica



Axiom: tai_chi_chuan type AtividadeAdequadaParaIdoso
1)   tai_chi_chuan temImpacto impacto_baixo
     AtividadeAdequadaParaIdoso equivalentTo AtividadeFisica
                                             and gastoMET some decimal[< 6.0]
                                             and temImpacto value impacto_baixo
     AtividadeDeFlexibilidade subClassOf AtividadeFisica
     tai_chi_chuan type AtividadeDeFlexibilidade
     tai_chi_chuan gastoMET 3.0


### 5.5 Interface de linha de comando

A CLI (`python recomendador_atividades.py`) faz as perguntas uma a uma. Aqui as respostas são passadas pela entrada padrão para registrar uma execução.

In [22]:
respostas = "\n".join(["Rafael", "45", "80", "1,78", "90", "8", "6,7", "3", "350", "indiferente"]) + "\n"
saida = subprocess.run([sys.executable, "recomendador_atividades.py"], input=respostas, text=True,
                       capture_output=True, check=True).stdout
perguntas, relatorio = saida.split("Usuário: Rafael")
print(perguntas)
print("Usuário: Rafael" + relatorio.split("Candidatas descartadas")[0])

Recomendador de atividades físicas (ontologia OWL 2 + HermiT)

Nome: Idade (anos): Peso (kg): Altura (m): Minutos de exercício por semana hoje: Há quantos meses pratica exercícios com regularidade: Objetivos:
  1. Ganhar flexibilidade
  2. Ganhar força
  3. Melhorar o condicionamento
  4. Perder peso
  5. Prevenir quedas
  6. Reduzir o estresse
  7. Socializar
Números separados por vírgula: Condições de saúde:
  1. Artrose no joelho
  2. Entorse recorrente de tornozelo
  3. Hipertensão não controlada
  4. Hérnia de disco
  5. Tendinite no ombro
Números separados por vírgula (0 = nenhuma, Enter = prefiro não informar): Orçamento mensal (R$): Ambiente preferido (ar livre / fechado / indiferente): 

Usuário: Rafael
Fatos informados:
  idade = 45
  IMC = 25.2
  minutosAtivosPorSemana = 90
  mesesDeExperiencia = 8
  temObjetivo = reduzir_estresse, socializar
  temCondicao = hipertensao_nao_controlada
  orçamento = R$ 350.00
  ambiente preferido = indiferente
Perfis inferidos pelo HermiT (0.

## 6. Comentários

O reasoner faz três trabalhos no sistema: classifica usuário e atividades em classes definidas, deduz relações que ninguém cadastrou (contraindicações e candidatas) e verifica a consistência da base. A parte procedural ficou pequena: validar a entrada, ler as inferências, aplicar os filtros e somar a pontuação. A análise de vantagens e limitações está no `README.md`.